# Comprehensive Text-to-Knowledge-Graph Pipeline
**Features:**
- Ingests text from Web URLs, PDF documents, or raw TXT files.
- Extracts Subject-Predicate-Object (SPO) triples from **complex sentences** using advanced spaCy dependency tree parsing.
- Prints extracted triples directly as formatted text and Pandas DataFrames.
- Displays an interactive **PyVis Knowledge Graph** rendered directly inside the Jupyter Notebook.

In [1]:
import sys
import os

# Install required packages
!{sys.executable} -m pip install spacy pandas pyvis requests PyPDF2 -q
!{sys.executable} -m spacy download en_core_web_sm -q

import re
import urllib.request
import urllib.parse
import json
import pandas as pd
import spacy
from PyPDF2 import PdfReader
from pyvis.network import Network
from IPython.display import display, HTML

# Load spaCy NLP Model
nlp = spacy.load("en_core_web_sm")
print("Environment successfully configured and spaCy model loaded!")

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
Environment successfully configured and spaCy model loaded!


In [2]:
import urllib.request
import urllib.parse
import re
from PyPDF2 import PdfReader

def load_input_text(source, source_type="text"):
    """
    Reads text from different input sources:
    - 'url'  : Web page URL or Wikipedia topic title
    - 'pdf'  : Path to a PDF file
    - 'txt'  : Path to a plain text file
    - 'text' : Raw text string
    """
    if source_type == "url":
        # Handle Wikipedia titles vs standard URLs
        if not source.startswith("http"):
            url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{urllib.parse.quote(source)}"
            req = urllib.request.Request(url, headers={'User-Agent': 'TripleExtractor/1.0'})
            try:
                with urllib.request.urlopen(req) as response:
                    data = json.loads(response.read().decode())
                    return data.get("extract", "")
            except Exception as e:
                print(f"Error fetching API URL: {e}")
                return ""
        else:
            try:
                req = urllib.request.Request(source, headers={'User-Agent': 'Mozilla/5.0'})
                with urllib.request.urlopen(req) as resp:
                    html_content = resp.read().decode('utf-8', errors='ignore')
                
                # Remove inline CSS and JS scripts
                html_content = re.sub(r'<(script|style)[^>]*>.*?</\1>', '', html_content, flags=re.DOTALL)
                # Extract text from paragraph tags
                paragraphs = re.findall(r'<p[^>]*>(.*?)</p>', html_content, flags=re.DOTALL)
                clean_paragraphs = [re.sub(r'<[^>]+>', '', p).strip() for p in paragraphs]
                text = "\n".join([p for p in clean_paragraphs if p])
                
                # Fallback to general tag stripping if no paragraphs found
                if not text:
                    text = re.sub(r'<[^>]+>', ' ', html_content)
                    text = re.sub(r'\s+', ' ', text).strip()
                return text
            except Exception as e:
                print(f"Error fetching web page: {e}")
                return ""

    elif source_type == "pdf":
        try:
            reader = PdfReader(source)
            text = ""
            for page in reader.pages:
                extracted = page.extract_text()
                if extracted:
                    text += extracted + "\n"
            return text.strip()
        except Exception as e:
            print(f"Error reading PDF file: {e}")
            return ""

    elif source_type == "txt":
        try:
            with open(source, "r", encoding="utf-8") as f:
                return f.read().strip()
        except Exception as e:
            print(f"Error reading TXT file: {e}")
            return ""

    return source

In [3]:
# # --- OPTION 1: RAW TEXT INPUT ---
# INPUT_DATA = (
#     "Mamallapuram, which is an ancient historic town situated in the Chengalpattu district of Tamil Nadu, "
#     "contains the famous Shore Temple. King Narasimhavarman I founded the town in the 7th century, "
#     "and he built several rock-cut monuments near the Bay of Bengal coast."
# )

# raw_text = load_input_text(INPUT_DATA, source_type="text")

# print("--- Loaded Raw Text Input ---")
# print(raw_text)

In [4]:
# # --- OPTION 2: WEB URL OR WIKIPEDIA ---
# # Pass a full HTTP link OR a Wikipedia topic name (e.g., "Knowledge_graph")
# INPUT_DATA = "https://en.wikipedia.org/wiki/Knowledge_graph"

# raw_text = load_input_text(INPUT_DATA, source_type="url")

# print("--- Loaded Web Content (First 500 characters) ---")
# print(raw_text[:500])

In [5]:
# --- OPTION 3: PDF FILE ---
# Specify the path to your PDF file
INPUT_DATA = "/home/jegan/Documents/knowledge graph/small.pdf"

raw_text = load_input_text(INPUT_DATA, source_type="pdf")

print("--- Loaded PDF Content (First 500 characters) ---")
print(raw_text[:500])

--- Loaded PDF Content (First 500 characters) ---
The following abbrevations are used in this document:
DLE means Discipline Linked Engineering Courses
DE means Discipline Elective
SE - Specialization Elective
OE means Open Elective
SEC means Skill Enhancement Courses
AEC means Ability Enhancement Courses
DC - Discipline Core
FC - Foundation core
LTPC - Lecture Tutorial Practical Credits
TH - Theory Only Courses
LO - Lab only courses
CO - Course Outcomes
FFCS - Fully Flexible Credit System
PBL - Project Based Learning
CAL - Curriculum for Appli


In [6]:
def extract_complex_triples(raw_text):
    """
    Extracts Subject-Predicate-Object triples from complex sentences
    by traversing dependency trees, handling conjunctions, prep phrases, and sub-clauses.
    """
    doc = nlp(raw_text)
    triples = []

    for sent in doc.sents:
        # Find all verbs/predicates in sentence (including auxiliary and clausal verbs)
        verbs = [token for token in sent if token.pos_ in ["VERB", "AUX"]]
        
        for verb in verbs:
            subjects = []
            objects = []

            # 1. Look for direct subjects connected to the verb
            for child in verb.children:
                if "subj" in child.dep_:
                    # Gather full noun phrase for subject
                    subj_tokens = [tok.text for tok in child.subtree if tok.dep_ not in ["punct", "cc", "conj"]]
                    subjects.append(" ".join(subj_tokens))

            # If verb is dependent on another verb (e.g., relative clause), borrow parent subject
            if not subjects and verb.dep_ in ["acl", "relcl", "xcomp", "advcl"]:
                for child in verb.head.children:
                    if "subj" in child.dep_:
                        subj_tokens = [tok.text for tok in child.subtree if tok.dep_ not in ["punct", "cc", "conj"]]
                        subjects.append(" ".join(subj_tokens))

            # 2. Look for objects or attribute complements
            for child in verb.children:
                if any(dep in child.dep_ for dep in ["obj", "attr", "acomp"]):
                    obj_tokens = [tok.text for tok in child.subtree if tok.dep_ not in ["punct", "cc", "conj"]]
                    objects.append(" ".join(obj_tokens))
                
                # Prepositional objects (e.g. "in Tamil Nadu")
                elif child.dep_ == "prep":
                    prep_pobj = [tok.text for tok in child.subtree if tok.dep_ not in ["punct", "cc", "conj"]]
                    if prep_pobj:
                        objects.append(" ".join(prep_pobj))

            # Construct triples for valid combinations
            for subj in subjects:
                for obj in objects:
                    if subj and verb.lemma_ and obj and (subj.lower() != obj.lower()):
                        triples.append({
                            "Subject": subj.strip(),
                            "Predicate": verb.lemma_.strip(),
                            "Object": obj.strip()
                        })

    # Drop duplicate triples
    df = pd.DataFrame(triples).drop_duplicates()
    return df

# Run Triple Extraction
triples_df = extract_complex_triples(raw_text)

print("=" * 80)
print("EXTRACTED TRIPLES (PRINTED OUTPUT)")
print("=" * 80)
for idx, row in triples_df.iterrows():
    print(f"[{idx+1:02d}] ({row['Subject']})  --[{row['Predicate']}]-->  ({row['Object']})")

print("\n" + "=" * 80)
print(" TRIPLES DATAFRAME")
print("=" * 80)
display(triples_df)

EXTRACTED TRIPLES (PRINTED OUTPUT)
[01] (The following abbrevations)  --[use]-->  (in this document)
[02] (Skill Enhancement Courses 
 AEC)  --[mean]-->  (Ability Enhancement Courses 
 DC Discipline Core 
 FC Foundation core 
 LTPC Lecture Tutorial Practical Credits 
 TH Theory Only Courses 
 LO Lab only courses 
 CO Course Outcomes 
 FFCS Fully Flexible Credit System 
 PBL Project)
[03] (Based Learning 
 CAL Curriculum for Applied Learning 
 ICT Information Technology 
 VTOP VIT on Top)  --[be]-->  (a rewritten document about FFCS regulation 4.0 of VIT university)
[04] (This)  --[be]-->  (a rewritten document about FFCS regulation 4.0 of VIT university)
[05] (This)  --[contain]-->  (details about the programmes offered in the university)
[06] (the terms program)  --[consider]-->  (In this document 
 pertaining to academia)
[07] (students)  --[make]-->  (decisions)
[08] (students)  --[make]-->  (on their own)
[09] (students)  --[have]-->  (multi - disciplinary competency)
[10] (Student

,Subject,Predicate,Object
0,The following abbrevations,use,in this document
1,Skill Enhancement Courses \n AEC,mean,Ability Enhancement Courses \n DC Discipline C...
2,Based Learning \n CAL Curriculum for Applied L...,be,a rewritten document about FFCS regulation 4.0...
3,This,be,a rewritten document about FFCS regulation 4.0...
4,This,contain,details about the programmes offered in the un...
...,...,...,...
106,they,bring,innovation
107,they,bring,in the product \n manufacturing industry
108,The students,approach,product \n design
109,The students,approach,from a holistic viewpoint integrating the aest...


In [7]:
# import urllib.request
# import urllib.parse
# import re

# def fetch_web_text(url):
#     req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
#     with urllib.request.urlopen(req) as response:
#         html = response.read().decode('utf-8', errors='ignore')
    
#     # Strip script and style blocks
#     html = re.sub(r'<(script|style)[^>]*>.*?</\1>', '', html, flags=re.DOTALL)
#     # Extract text content inside paragraph tags
#     paragraphs = re.findall(r'<p[^>]*>(.*?)</p>', html, flags=re.DOTALL)
#     # Clean HTML tags from extracted text
#     clean_paragraphs = [re.sub(r'<[^>]+>', '', p).strip() for p in paragraphs]
    
#     return "\n".join([p for p in clean_paragraphs if p])

# # Test URL
# target_url = "https://en.wikipedia.org/wiki/Knowledge_graph"
# raw_text = fetch_web_text(target_url)
# print("--- Text Extracted Successfully ---")
# print(raw_text[:500])

In [8]:
import os
from IPython.display import IFrame, display

def visualize_knowledge_graph(df, output_html="knowledge_graph.html"):
    """
    Constructs a PyVis interactive graph and renders it securely inside Jupyter using an IFrame.
    """
    if df.empty:
        print("⚠️ No triples extracted to display graph.")
        return

    # Create PyVis Network Graph
    net = Network(notebook=True, cdn_resources='remote', height="550px", width="100%", directed=True)

    # Add Nodes and Edges
    for _, row in df.iterrows():
        subj = str(row['Subject'])
        pred = str(row['Predicate'])
        obj = str(row['Object'])

        # Subject node (Blue)
        net.add_node(subj, label=subj, color="#3498DB", shape="dot", size=22)
        # Object node (Orange)
        net.add_node(obj, label=obj, color="#E67E22", shape="dot", size=22)
        # Directed Edge (Red text/line)
        net.add_edge(subj, obj, title=pred, label=pred, color="#E74C3C")

    # Configure physics for better node layout
    net.toggle_physics(True)
    
    # Generate the standalone HTML file
    net.write_html(output_html)

    # Display using IFrame to prevent script blocking
    display(IFrame(src=output_html, width="100%", height="580"))

# Display Graph
visualize_knowledge_graph(triples_df)